# 📄 PDF → PDF con OCR buscable

### Para usarlo:
1. Tocá **▶** una sola vez.
2. Colab va a mostrar una **barra real de preparación** que avanza por las etapas completadas.
3. Cuando aparezca **Elegir archivos**, subí tu PDF.
4. Durante el OCR vas a ver otra barra real: **`OCR del PDF 37/142 páginas`**, con porcentaje, velocidad y tiempo estimado.
5. Al terminar, se descarga automáticamente `nombre_OCR.pdf`.

**No tenés que editar código.**  
Si la barra cambia o aumenta el contador de páginas, está trabajando.

El PDF original no se modifica: se crea una copia visualmente igual con una capa de texto invisible para poder usar **Ctrl+F**, seleccionar y copiar texto.

In [ ]:
# ============================================================
# 📄 PDF → PDF CON OCR BUSCABLE
# Tocá ▶ una sola vez. Después, seguí lo que aparece en pantalla.
# ============================================================

import os, sys, subprocess, shutil, pathlib, zipfile, time, html
from IPython.display import display, HTML
from tqdm.auto import tqdm

# ------------------------------------------------------------
# PANEL VISUAL DE ESTADO
# ------------------------------------------------------------

started_at = time.time()

def fmt_time(seconds):
    seconds = max(0, int(seconds))
    m, s = divmod(seconds, 60)
    h, m = divmod(m, 60)
    if h:
        return f"{h} h {m:02d} min"
    if m:
        return f"{m} min {s:02d} s"
    return f"{s} s"

def progress_card(
    title,
    detail="",
    percent=None,
    stage="Trabajando",
    current=None,
    total=None,
    eta=None,
    status="working",
):
    if percent is None:
        bar_width = 100
        bar_class = "indeterminate"
        pct_label = ""
    else:
        percent = max(0, min(100, float(percent)))
        bar_width = percent
        bar_class = ""
        pct_label = f"{percent:.0f}%"

    icons = {
        "working": "⚙️",
        "ok": "✅",
        "error": "❌",
        "upload": "⬆️",
        "download": "⬇️",
    }
    icon = icons.get(status, "⚙️")

    count_html = ""
    if current is not None and total is not None:
        count_html = f"""
        <div class="ocr-count">
          <strong>{html.escape(str(current))}</strong>
          <span>de</span>
          <strong>{html.escape(str(total))}</strong>
        </div>
        """

    eta_html = ""
    if eta is not None and eta >= 0:
        eta_html = f"""
        <div class="ocr-meta-item">
          <span>Tiempo restante aprox.</span>
          <strong>{html.escape(fmt_time(eta))}</strong>
        </div>
        """

    elapsed = time.time() - started_at

    return HTML(f"""
    <style>
      .ocr-card {{
        font-family: Inter, Arial, sans-serif;
        max-width: 760px;
        padding: 22px;
        border: 1px solid #dadce0;
        border-radius: 18px;
        background: #fff;
        box-shadow: 0 4px 18px rgba(0,0,0,.06);
        color: #202124;
      }}
      .ocr-top {{
        display: flex;
        gap: 14px;
        align-items: flex-start;
      }}
      .ocr-icon {{
        font-size: 30px;
        line-height: 1;
      }}
      .ocr-stage {{
        margin: 0 0 4px;
        color: #5f6368;
        font-size: 12px;
        font-weight: 800;
        letter-spacing: .06em;
        text-transform: uppercase;
      }}
      .ocr-title {{
        margin: 0;
        font-size: 21px;
        line-height: 1.25;
      }}
      .ocr-detail {{
        margin: 7px 0 0;
        color: #5f6368;
        font-size: 14px;
        line-height: 1.55;
      }}
      .ocr-progress-wrap {{
        margin-top: 20px;
      }}
      .ocr-progress-head {{
        display: flex;
        justify-content: space-between;
        align-items: center;
        margin-bottom: 8px;
        color: #5f6368;
        font-size: 13px;
      }}
      .ocr-progress {{
        position: relative;
        height: 13px;
        overflow: hidden;
        border-radius: 999px;
        background: #eceff3;
      }}
      .ocr-progress-bar {{
        height: 100%;
        width: {bar_width}%;
        border-radius: 999px;
        background: linear-gradient(90deg, #1a73e8, #6aa7ff);
        transition: width .25s ease;
      }}
      .ocr-progress-bar.indeterminate {{
        width: 35%;
        animation: ocr-slide 1.15s infinite ease-in-out;
      }}
      @keyframes ocr-slide {{
        0%   {{ transform: translateX(-120%); }}
        100% {{ transform: translateX(320%); }}
      }}
      .ocr-bottom {{
        display: flex;
        flex-wrap: wrap;
        align-items: center;
        gap: 10px 18px;
        margin-top: 16px;
      }}
      .ocr-count {{
        display: inline-flex;
        align-items: baseline;
        gap: 5px;
        padding: 8px 11px;
        border-radius: 10px;
        background: #f1f3f4;
      }}
      .ocr-count strong {{
        font-size: 17px;
      }}
      .ocr-count span {{
        color: #5f6368;
        font-size: 12px;
      }}
      .ocr-meta-item {{
        display: flex;
        flex-direction: column;
        gap: 2px;
      }}
      .ocr-meta-item span {{
        color: #80868b;
        font-size: 11px;
      }}
      .ocr-meta-item strong {{
        font-size: 13px;
      }}
      .ocr-alive {{
        margin-left: auto;
        color: #188038;
        font-size: 12px;
        font-weight: 700;
      }}
      @media (prefers-color-scheme: dark) {{
        .ocr-card {{
          background: #202124;
          border-color: #3c4043;
          color: #e8eaed;
        }}
        .ocr-stage, .ocr-detail, .ocr-progress-head {{
          color: #bdc1c6;
        }}
        .ocr-progress {{
          background: #3c4043;
        }}
        .ocr-count {{
          background: #303134;
        }}
        .ocr-count span, .ocr-meta-item span {{
          color: #9aa0a6;
        }}
      }}
    </style>

    <div class="ocr-card">
      <div class="ocr-top">
        <div class="ocr-icon">{icon}</div>
        <div style="flex:1">
          <div class="ocr-stage">{html.escape(stage)}</div>
          <h3 class="ocr-title">{html.escape(title)}</h3>
          <div class="ocr-detail">{html.escape(detail)}</div>
        </div>
      </div>

      <div class="ocr-progress-wrap">
        <div class="ocr-progress-head">
          <span>Progreso</span>
          <strong>{pct_label}</strong>
        </div>
        <div class="ocr-progress">
          <div class="ocr-progress-bar {bar_class}"></div>
        </div>
      </div>

      <div class="ocr-bottom">
        {count_html}
        <div class="ocr-meta-item">
          <span>Tiempo transcurrido</span>
          <strong>{fmt_time(elapsed)}</strong>
        </div>
        {eta_html}
        <div class="ocr-alive">● Sigue trabajando · este panel se actualiza solo</div>
      </div>
    </div>
    """)

panel = display(
    progress_card(
        "Preparando el entorno",
        "Estamos configurando todo automáticamente. No cierres esta pestaña.",
        stage="Paso 1 de 5",
    ),
    display_id=True,
)

def update_panel(*args, **kwargs):
    panel.update(progress_card(*args, **kwargs))

def run_live(cmd, title, detail, stage, substep=None, substeps=None):
    """
    Ejecuta un comando externo sin congelar el panel.
    Mientras el proceso sigue vivo, actualiza el tiempo cada segundo.
    """
    suffix = ""
    if substep is not None and substeps is not None:
        suffix = f" · instalación {substep}/{substeps}"

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    pulse = 0
    while proc.poll() is None:
        pulse += 1
        dots = "." * ((pulse % 3) + 1)
        update_panel(
            title,
            f"{detail}{dots}",
            stage=f"{stage}{suffix}",
            percent=None,
            status="working",
        )
        time.sleep(1)

    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)

def package_importable(module_name):
    """Chequeo barato para no reinstalar paquetes que ya están disponibles."""
    try:
        __import__(module_name)
        return True
    except Exception:
        return False

def paddle_gpu_ready():
    try:
        import paddle
        return bool(
            paddle.is_compiled_with_cuda()
            and paddle.device.cuda.device_count() >= 1
        )
    except Exception:
        return False

# ------------------------------------------------------------
# 1) INSTALACIÓN
# ------------------------------------------------------------

# Barra NATIVA de Colab/Jupyter. Avanza por tareas reales completadas.
setup_bar = tqdm(
    total=4,
    desc="Preparando OCR",
    unit="etapa",
    dynamic_ncols=True,
    leave=True,
)

setup_bar.set_postfix_str("Comprobando PaddlePaddle / CUDA")

# 1/4 — Paddle GPU
if not paddle_gpu_ready():
    run_live(
        [sys.executable, "-m", "pip", "uninstall", "-y", "-q",
         "paddlepaddle", "paddlepaddle-gpu"],
        "Preparando PaddlePaddle",
        "Quitando una versión incompatible si existiera",
        "Paso 1 de 5",
        1, 3,
    )

    run_live(
        [sys.executable, "-m", "pip", "install", "-q",
         "paddlepaddle-gpu==3.3.0",
         "-i", "https://www.paddlepaddle.org.cn/packages/stable/cu126/"],
        "Instalando el motor CUDA",
        "Esta suele ser la parte más lenta. Colab sigue trabajando",
        "Paso 1 de 5",
        2, 3,
    )

setup_bar.update(1)
setup_bar.set_postfix_str("Comprobando PaddleOCR y PDF")

# 2/4 — OCR + utilidades PDF
missing_modules = [
    name for name in ("paddleocr", "pymupdf", "PIL")
    if not package_importable(name)
]

if missing_modules:
    run_live(
        [sys.executable, "-m", "pip", "install", "-q", "-U",
         "paddleocr", "pymupdf", "pillow"],
        "Instalando PaddleOCR y herramientas PDF",
        "Última parte de la preparación",
        "Paso 1 de 5",
        3, 3,
    )

setup_bar.update(1)
setup_bar.set_postfix_str("Importando librerías")

# ------------------------------------------------------------
# 2) GPU + MODELOS
# ------------------------------------------------------------

update_panel(
    "Comprobando la GPU",
    "Colab está verificando que pueda usar CUDA para acelerar el OCR.",
    stage="Paso 2 de 5",
)

import numpy as np
import paddle
import pymupdf
from PIL import Image
from paddleocr import PaddleOCR
from google.colab import files

setup_bar.update(1)
setup_bar.set_postfix_str("Verificando GPU")

if (not paddle.is_compiled_with_cuda()) or paddle.device.cuda.device_count() < 1:
    panel.update(HTML("""
    <div style="
      max-width:760px;padding:22px;border:2px solid #d93025;border-radius:16px;
      font-family:Arial;background:#fff3f2;color:#202124">
      <h3 style="margin-top:0">❌ Falta activar la GPU</h3>
      <p>No se rompió nada. Solo falta cambiar una opción de Colab.</p>
      <ol style="line-height:1.7">
        <li>Arriba, abrí <b>Entorno de ejecución</b>.</li>
        <li>Elegí <b>Cambiar tipo de entorno de ejecución</b>.</li>
        <li>En acelerador de hardware elegí <b>GPU</b>.</li>
        <li>Guardá y volvé a tocar ▶.</li>
      </ol>
    </div>
    """))
    raise RuntimeError("GPU no disponible.")

paddle.set_device("gpu:0")

try:
    gpu_name = paddle.device.cuda.get_device_name()
except Exception:
    gpu_name = "GPU CUDA"

update_panel(
    "Cargando los modelos de reconocimiento",
    f"GPU detectada: {gpu_name}. La primera carga puede tardar porque descarga los modelos.",
    stage="Paso 2 de 5",
)

from concurrent.futures import ThreadPoolExecutor

def build_ocr():
    return PaddleOCR(
        lang="es",
        device="gpu:0",
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=True,
    )

with ThreadPoolExecutor(max_workers=1) as executor:
    future = executor.submit(build_ocr)
    pulse = 0

    while not future.done():
        pulse += 1
        dots = "." * ((pulse % 3) + 1)
        update_panel(
            "Cargando los modelos de reconocimiento",
            f"GPU detectada: {gpu_name}. Descargando o preparando modelos{dots}",
            stage="Paso 2 de 5",
            percent=None,
            status="working",
        )
        time.sleep(1)

    ocr = future.result()

update_panel(
    "Modelos cargados",
    f"GPU detectada: {gpu_name}. Todo listo para recibir el PDF.",
    stage="Paso 2 de 5",
    percent=100,
    status="ok",
)
time.sleep(0.8)

setup_bar.update(1)
setup_bar.set_postfix_str("Listo ✓")
setup_bar.close()

# ------------------------------------------------------------
# 3) SUBIDA
# ------------------------------------------------------------

DPI = 200
MIN_SCORE = 0.35

FONT_PATH = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
if not os.path.exists(FONT_PATH):
    run_live(
        ["apt-get", "update", "-qq"],
        "Preparando una fuente compatible",
        "Actualizando la lista de paquetes",
        "Paso 2 de 5",
    )
    run_live(
        ["apt-get", "install", "-y", "-qq", "fonts-dejavu-core"],
        "Preparando una fuente compatible",
        "Instalando la fuente para la capa invisible del PDF",
        "Paso 2 de 5",
    )

font_for_measure = pymupdf.Font(fontfile=FONT_PATH)

update_panel(
    "Elegí tu PDF",
    "Aparecerá el selector de archivos debajo. Podés subir uno o varios PDFs.",
    stage="Paso 3 de 5",
    percent=0,
    status="upload",
)

uploaded = files.upload()

pdfs = []
for filename in uploaded:
    path = pathlib.Path("/content") / filename
    if path.suffix.lower() == ".pdf":
        pdfs.append(path)

if not pdfs:
    panel.update(HTML("""
    <div style="
      max-width:760px;padding:22px;border:2px solid #d93025;border-radius:16px;
      font-family:Arial;background:#fff3f2;color:#202124">
      <h3 style="margin-top:0">❌ No encontré un PDF</h3>
      <p>Volvé a tocar ▶ y elegí un archivo terminado en <b>.pdf</b>.</p>
    </div>
    """))
    raise ValueError("No se seleccionó ningún archivo PDF.")

def result_dict(res):
    data = getattr(res, "json", res)
    if callable(data):
        data = data()
    if isinstance(data, dict) and "res" in data:
        data = data["res"]
    return data

def add_hidden_text(page, texts, scores, boxes, img_w, img_h):
    if img_w <= 0 or img_h <= 0:
        return 0

    sx = page.rect.width / float(img_w)
    sy = page.rect.height / float(img_h)

    fontname = "dejavu"
    page.insert_font(fontname=fontname, fontfile=FONT_PATH)

    inserted = 0

    for text, score, box in zip(texts, scores, boxes):
        text = str(text).strip()
        if not text or float(score) < MIN_SCORE:
            continue

        x0, y0, x1, y1 = [float(v) for v in box]
        x0 *= sx; x1 *= sx
        y0 *= sy; y1 *= sy

        x0 = max(0.0, min(x0, page.rect.width))
        x1 = max(0.0, min(x1, page.rect.width))
        y0 = max(0.0, min(y0, page.rect.height))
        y1 = max(0.0, min(y1, page.rect.height))

        width = max(1.0, x1 - x0)
        height = max(1.0, y1 - y0)

        try:
            unit_width = max(font_for_measure.text_length(text, fontsize=1), 0.01)
        except Exception:
            unit_width = max(len(text) * 0.55, 0.01)

        by_height = height * 0.80
        by_width = width / unit_width * 0.97
        fontsize = max(2.5, min(by_height, by_width))
        baseline = y0 + min(height * 0.82, fontsize * 1.08)

        try:
            page.insert_text(
                pymupdf.Point(x0, baseline),
                text,
                fontname=fontname,
                fontsize=fontsize,
                render_mode=3,
                overlay=True,
            )
            inserted += 1
        except Exception:
            pass

    return inserted

# Contar páginas antes de empezar para tener una barra GLOBAL real.
pdf_page_counts = {}
total_pages_all = 0

for pdf in pdfs:
    temp_doc = pymupdf.open(str(pdf))
    n = len(temp_doc)
    temp_doc.close()
    pdf_page_counts[pdf] = n
    total_pages_all += n

processed_global = 0
global_start = time.time()
outputs = []

# ------------------------------------------------------------
# 4) OCR
# ------------------------------------------------------------

# Esta barra sí representa progreso REAL: 1 unidad = 1 página terminada.
ocr_progress = tqdm(
    total=total_pages_all,
    desc="OCR del PDF",
    unit="pág",
    dynamic_ncols=True,
    leave=True,
)

for file_idx, pdf in enumerate(pdfs, start=1):
    out = pathlib.Path("/content") / f"{pdf.stem}_OCR.pdf"
    doc = pymupdf.open(str(pdf))
    pages_in_file = len(doc)
    total_lines = 0

    for page_idx in range(pages_in_file):
        page_start = time.time()
        page = doc[page_idx]

        ocr_progress.set_postfix_str(
            f"{pdf.name} · página {page_idx + 1}/{pages_in_file}"
        )

        # Progreso ANTES de procesar la página actual.
        pct = (processed_global / total_pages_all) * 100 if total_pages_all else 0

        elapsed_processing = time.time() - global_start
        avg_page = elapsed_processing / processed_global if processed_global > 0 else None
        remaining_pages = total_pages_all - processed_global
        eta = avg_page * remaining_pages if avg_page else None

        update_panel(
            f"Reconociendo texto · {pdf.name}",
            f"Archivo {file_idx} de {len(pdfs)} · Página {page_idx + 1} de {pages_in_file}",
            stage="Paso 4 de 5 · OCR",
            percent=pct,
            current=processed_global + 1,
            total=total_pages_all,
            eta=eta,
            status="working",
        )

        if page.rotation:
            page.remove_rotation()

        pix = page.get_pixmap(
            dpi=DPI,
            colorspace=pymupdf.csRGB,
            alpha=False
        )

        img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
        arr = np.asarray(img)
        results = ocr.predict(arr)

        for res in results:
            data = result_dict(res)
            texts = list(data.get("rec_texts", []))
            scores = list(data.get("rec_scores", []))
            boxes = data.get("rec_boxes", [])

            total_lines += add_hidden_text(
                page, texts, scores, boxes, pix.width, pix.height
            )

        processed_global += 1
        ocr_progress.update(1)

        # Actualizar al terminar cada página para que la barra avance realmente.
        pct = (processed_global / total_pages_all) * 100
        elapsed_processing = time.time() - global_start
        avg_page = elapsed_processing / processed_global
        remaining_pages = total_pages_all - processed_global
        eta = avg_page * remaining_pages

        update_panel(
            f"Reconociendo texto · {pdf.name}",
            f"Página {page_idx + 1} de {pages_in_file} terminada · {total_lines} bloques de texto agregados en este archivo",
            stage="Paso 4 de 5 · OCR",
            percent=pct,
            current=processed_global,
            total=total_pages_all,
            eta=eta,
            status="working",
        )

    update_panel(
        f"Guardando {pdf.name}",
        "El OCR ya terminó para este archivo. Ahora estamos construyendo el PDF final sin cambiar el aspecto de las páginas.",
        stage="Paso 4 de 5 · Guardando",
        percent=(processed_global / total_pages_all) * 100,
        current=processed_global,
        total=total_pages_all,
        status="working",
    )

    doc.save(
        str(out),
        garbage=4,
        deflate=True,
        clean=True
    )
    doc.close()
    outputs.append(out)

ocr_progress.set_postfix_str("OCR terminado ✓")
ocr_progress.close()

# ------------------------------------------------------------
# 5) DESCARGA
# ------------------------------------------------------------

if len(outputs) == 1:
    update_panel(
        "¡Listo! Preparando la descarga",
        f"Se creó {outputs[0].name}. La descarga debería comenzar automáticamente.",
        stage="Paso 5 de 5",
        percent=100,
        current=total_pages_all,
        total=total_pages_all,
        eta=0,
        status="download",
    )
    files.download(str(outputs[0]))

else:
    zip_path = "/content/PDFs_con_OCR.zip"

    update_panel(
        "Empaquetando los PDFs",
        f"Procesamos {len(outputs)} archivos. Los estamos juntando en un ZIP para descargarlos de una sola vez.",
        stage="Paso 5 de 5",
        percent=100,
        current=total_pages_all,
        total=total_pages_all,
        eta=0,
        status="download",
    )

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for p in outputs:
            z.write(p, arcname=p.name)

    files.download(zip_path)

# Mensaje final.
panel.update(HTML(f"""
<div style="
  max-width:760px;padding:24px;border:2px solid #188038;border-radius:18px;
  font-family:Inter,Arial;background:#edf7ed;color:#1e4620">
  <div style="font-size:30px">✅</div>
  <h2 style="margin:8px 0">OCR terminado</h2>
  <p style="margin:0 0 8px;line-height:1.55">
    Procesamos <b>{total_pages_all} páginas</b>.
    El PDF mantiene su aspecto original y ahora tiene texto seleccionable y buscable.
  </p>
  <p style="margin:0;color:#4b6350">
    Tiempo total: <b>{fmt_time(time.time() - started_at)}</b>.
    Ya podés cerrar esta pestaña cuando termine la descarga.
  </p>
</div>
"""))